# Lesson 23: Did Dream Get Lucky?

**2020.** The YouTuber Dream submitted a world-record Minecraft speedrun. Two random events decide how fast a run can go:

- Trading gold to a **piglin** gives ender pearls with probability **0.047**.
- Killing a **blaze** drops a blaze rod with probability **0.50**.

The game's code sets those chances. Speedrun moderators went through Dream's streams and counted his results. Was he just lucky, or was something wrong with his game?

$$H_0: p = 0.047 \quad \text{(the game was fair)} \qquad H_a: p > 0.047 \quad \text{(his pearl rate was higher than the game allows)}$$

Run each cell in order (Shift + Enter).

> Colab setup: *Runtime → Change runtime type → R*.

## 1. The data

First, how many successes would an honest player expect? Expected count $= n \times p_0$.

In [ ]:
library(ggplot2)
set.seed(206)
options(repr.plot.width = 12, repr.plot.height = 7, repr.plot.res = 120)

# What the moderators counted across Dream's streams
pearl_tries <- 262;  pearl_hits <- 42;   pearl_p0 <- 0.047   # piglin trades that gave pearls
rod_tries   <- 305;  rod_hits   <- 211;  rod_p0   <- 0.50    # blazes that dropped a rod

# Expected successes for an honest player
cat("Pearls: expected", round(pearl_tries * pearl_p0, 1), " Dream got", pearl_hits, "\n")
cat("Rods:   expected", round(rod_tries * rod_p0, 1),   " Dream got", rod_hits, "\n")

**What to notice:** Dream got more than three times the pearls an honest player expects.

## 2. The test statistic

$$z = \frac{\hat{p} - p_0}{\sqrt{p_0(1-p_0)/n}}$$

The standard error uses $p_0$, not $\hat{p}$, because the test starts by assuming $H_0$ is true.

In [ ]:
# Test statistic for each: z = (p-hat - p0) / sqrt(p0 (1 - p0) / n)
test_stat <- function(hits, n, p0) {
  p_hat <- hits / n
  se    <- sqrt(p0 * (1 - p0) / n)      # uses p0, because we assume H0 is true
  (p_hat - p0) / se
}

z_pearl <- test_stat(pearl_hits, pearl_tries, pearl_p0)
z_rod   <- test_stat(rod_hits,   rod_tries,   rod_p0)

cat("Pearls: p-hat =", round(pearl_hits / pearl_tries, 3), "  test statistic =", round(z_pearl, 2),
    "  P-value =", signif(pnorm(z_pearl, lower.tail = FALSE), 2), "\n")
cat("Rods:   p-hat =", round(rod_hits / rod_tries, 3),     "  test statistic =", round(z_rod, 2),
    "  P-value =", signif(pnorm(z_rod, lower.tail = FALSE), 2), "\n")

**What to notice:** both P-values are essentially zero. For reference, $1 \times 10^{-11}$ means 0.00000000001.

## 3. What do honest players look like?

The P-value asks: *if the game were fair, how often would someone do this well?* Instead of trusting a formula, let's build 100,000 honest players and see.

In [ ]:
# Simulate 100,000 HONEST players, each making 262 piglin trades
reps <- 100000
honest_pearls <- rbinom(reps, size = pearl_tries, prob = pearl_p0)

cat("Most pearls any honest player got:", max(honest_pearls), "   Dream got:", pearl_hits, "\n")

ggplot(data.frame(pearls = honest_pearls), aes(pearls)) +
  geom_bar(fill = "grey65") +
  geom_vline(xintercept = pearl_hits, color = "firebrick", linewidth = 1.5) +
  annotate("text", x = pearl_hits - 0.5, y = Inf, vjust = 2, hjust = 1, size = 5.5,
           color = "firebrick", label = "Dream: 42") +
  scale_x_continuous(limits = c(0, 45)) +
  labs(title = "Pearl trades for 100,000 honest players (262 trades each)",
       subtitle = "Not one honest player came close to Dream",
       x = "number of successful pearl trades", y = "number of players") +
  theme_minimal(base_size = 16) +
  theme(plot.background = element_rect(fill = "white", color = NA))

**What to notice:** the honest players pile up around 12, and the luckiest of 100,000 got about 29. Dream's 42 isn't in the tail of the distribution. It's off the chart entirely.

## 4. The same thing on the test statistic scale

Convert every honest player's result into a test statistic. If the large-sample conditions hold, these should follow the standard normal curve.

In [ ]:
# Turn every honest player's result into a test statistic.
# The CLT says these should follow the standard normal curve.
sims <- data.frame(z = test_stat(honest_pearls, pearl_tries, pearl_p0))
step <- 1 / (pearl_tries * sqrt(pearl_p0 * (1 - pearl_p0) / pearl_tries))  # one more pearl moves z by this much

ggplot(sims, aes(z)) +
  geom_histogram(aes(y = after_stat(density)), binwidth = step,
                 center = test_stat(0, pearl_tries, pearl_p0),
                 fill = "grey70", color = "white") +
  geom_function(fun = dnorm, linewidth = 1.2) +
  geom_vline(xintercept = z_pearl, color = "firebrick", linewidth = 1.5) +
  annotate("text", x = z_pearl - 0.2, y = 0.3, hjust = 1, size = 5.5, color = "firebrick",
           label = paste0("Dream's test statistic: ", round(z_pearl, 1))) +
  geom_vline(xintercept = 3.5, linetype = "dotted", linewidth = 1) +
  annotate("text", x = 3.6, y = 0.15, hjust = 0, size = 5,
           label = "the z table\nends here") +
  scale_x_continuous(breaks = -4:9) +
  coord_cartesian(xlim = c(-4, 9.5)) +
  labs(title = "Test statistics for 100,000 honest players",
       subtitle = "Curve: the standard normal. Dream is more than 8 standard errors past the claim.",
       x = "test statistic", y = "density") +
  theme_minimal(base_size = 16) +
  theme(plot.background = element_rect(fill = "white", color = NA))

**What to notice:** the honest players follow the normal curve, just as the large-sample theory promises. The $z$ table stops around 3.5 because almost nothing happens past there. Dream's test statistic is over 8.

## 5. Both events at once

Dream was lucky on pearls **and** on blaze rods. Plot every honest player's pearls against their rods.

In [ ]:
# Both events at once: each grey dot is one honest player's pearls AND rods.
honest_rods <- rbinom(reps, size = rod_tries, prob = rod_p0)
both <- data.frame(pearls = honest_pearls, rods = honest_rods)

ggplot(both, aes(pearls, rods)) +
  geom_jitter(width = 0.3, height = 0.3, alpha = 0.08, size = 1, color = "grey40") +
  annotate("point", x = pearl_hits, y = rod_hits, color = "firebrick", size = 6) +
  annotate("text", x = pearl_hits, y = rod_hits - 6, color = "firebrick", size = 5.5,
           label = "Dream") +
  labs(title = "100,000 honest players vs. Dream",
       subtitle = "Each grey dot is one honest player. Dream is off on his own.",
       x = "pearl trades (out of 262)", y = "blaze rods (out of 305)") +
  theme_minimal(base_size = 16) +
  theme(plot.background = element_rect(fill = "white", color = NA))

**What to notice:** being lucky on one event is rare. Being that lucky on both at the same time is far rarer. The moderators estimated the chance of both at about **1 in 7.5 trillion**, and that was after they adjusted for having picked Dream's luckiest streams.

## 6. Why the conditions matter

The test needs $np_0 \geq 10$ and $n(1-p_0) \geq 10$. Dream's pearls pass: $262 \times 0.047 = 12.3$. What if a player had made only 20 trades?

In [ ]:
# Why we need n*p0 >= 10 and n*(1 - p0) >= 10.
# Suppose a player made only 20 trades: n*p0 = 20 * 0.047 = 0.94, far below 10.
small_n <- 20
small <- data.frame(pearls = rbinom(reps, size = small_n, prob = pearl_p0))
small_curve <- data.frame(x = seq(-2, 5, by = 0.05))
small_curve$y <- dnorm(small_curve$x, mean = small_n * pearl_p0,
                       sd = sqrt(small_n * pearl_p0 * (1 - pearl_p0))) * reps

ggplot(small, aes(pearls)) +
  geom_bar(fill = "grey65") +
  geom_vline(xintercept = 0, linetype = "dashed") +
  geom_line(data = small_curve, aes(x, y), linewidth = 1.2) +
  labs(title = "Only 20 trades: the counts are lopsided, not bell-shaped",
       subtitle = "Bars: what really happens.  Curve: what the z test assumes. It even spills below zero.",
       x = "number of successful pearl trades (out of 20)", y = "number of players") +
  theme_minimal(base_size = 16) +
  theme(plot.background = element_rect(fill = "white", color = NA))

**What to notice:** with $np_0$ under 1, most players get 0 or 1 pearl, and the counts are lopsided. The normal curve the $z$ test relies on doesn't match the bars, and it even puts probability below zero pearls, which is impossible. That's what the conditions protect against.

## 7. The error to avoid

A common mistake is putting $\hat{p}$ in the standard error. Watch what it does.

In [ ]:
# The error to avoid: using p-hat in the standard error instead of p0
p_hat <- pearl_hits / pearl_tries
se_right <- sqrt(pearl_p0 * (1 - pearl_p0) / pearl_tries)
se_wrong <- sqrt(p_hat * (1 - p_hat) / pearl_tries)
cat("Standard error with p0 (correct):", round(se_right, 4),
    "  test statistic =", round((p_hat - pearl_p0) / se_right, 2), "\n")
cat("Standard error with p-hat (wrong):", round(se_wrong, 4),
    "  test statistic =", round((p_hat - pearl_p0) / se_wrong, 2), "\n")

**What to notice:** using $\hat{p}$ gives the wrong standard error and a different test statistic. The conclusion happens to survive here because Dream's result is so extreme, but in a closer case this mistake can flip the decision.

## 8. R's built-in test

`prop.test` does the whole test. Add `correct = FALSE` to match the hand calculation. R reports `X-squared`, which is just the test statistic squared: $8.67^2 \approx 75.1$.

In [ ]:
# R's built-in test. correct = FALSE matches the hand calculation.
prop.test(pearl_hits, pearl_tries, p = pearl_p0, alternative = "greater", correct = FALSE)
prop.test(rod_hits,   rod_tries,   p = rod_p0,   alternative = "greater", correct = FALSE)

### Conclusion
At any reasonable $\alpha$, reject $H_0$ for both events. The data give overwhelming evidence that Dream's pearl and rod rates were higher than the game allows. In 2021, Dream acknowledged he had been running a modified version of the game, though he said he did not know it.

### Questions to explore
1. In the first code cell, change `pearl_p0 <- 0.047` to `0.10` and rerun everything. Is Dream still suspicious if pearls were twice as common?
2. Dream's defense was that the moderators picked his luckiest streams. Why does cherry-picking make a P-value look smaller than it should be?
3. In Section 6, try `small_n <- 250`. Does the normal curve fit the bars now? Compute $np_0$ for that case.
4. A friend opens 50 loot boxes advertised at a 2% legendary rate and gets 3 legendaries. Can you run a one-proportion $z$ test? Check the conditions first.